<a href="https://colab.research.google.com/github/maiap22/Projeto-PLN/blob/main/Projeto_PLN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Tema do Projeto: Apresentação dos 5 artigos mais visitados de um país, com a tradução de seu título e o sentimento geral e temas.


###Técnicas usadas:

-Normalização das entradas

-LLMs(Gemini e GNews)

-Extração de dados de urls

-Sumarização de textos(Langchain)

-Classificação de textos(sentimentos)




*Instalação*

In [21]:
# Instala as bibliotecas necessárias
!pip install langchain langchain-google-genai langchain-openai gnews python-dotenv pydantic

print("Dependências instaladas com sucesso!")

Dependências instaladas com sucesso!


*Configuração das API Keys*

In [27]:
import os
import getpass # Para pedir a senha de forma segura, sem exibi-la

# --- Configuração das Chaves de API ---
# Utilizo 'getpass' para inserir suas chaves de API de forma segura.
# Isso evita que as chaves fiquem visíveis no histórico do notebook ou salvas em arquivos.
# Quando executar esta célula, uma caixa de texto aparecerá no Colab.
# Cole sua chave e pressione Enter para cada uma.

print("Cole sua chave de API do Google (Gemini) e pressione Enter:")
os.environ["GOOGLE_API_KEY"] = getpass.getpass()

print("Cole sua chave de API do GNews e pressione Enter:")
os.environ["GNEWS_API_KEY"] = getpass.getpass()

print("\nChaves de API configuradas!")

Cole sua chave de API do Google (Gemini) e pressione Enter:
··········
Cole sua chave de API do GNews e pressione Enter:
··········

Chaves de API configuradas!


*Carregamento de dados*

In [32]:
from gnews import GNews
import json
import os

articles_list = [] # Initialize articles_list to an empty list

# --- 1. Inputs do Usuário (Configuração Dinâmica) ---
print("--- 🌍 Configuração da Busca de Notícias ---")

# Pergunta ao usuário (se der Enter vazio, usa um padrão)
pais_input = input("Digite o código do país (ex: 'us', 'jp', 'br', 'fr') [padrão: us]: ").lower().strip()
if not pais_input: pais_input = 'us'

lingua_input = input("Digite o código do idioma (ex: 'en', 'ja', 'pt', 'fr') [padrão: en]: ").lower().strip()
if not lingua_input: lingua_input = 'en'

# --- 2. Mapeamento para o Prompt do LLM ---
# Traduz o código 'ja' para 'Japonês', para ajudar o LLM na Célula 6
mapa_idiomas = {
    'en': 'Inglês', 'pt': 'Português', 'ja': 'Japonês',
    'fr': 'Francês', 'de': 'Alemão', 'es': 'Espanhol',
    'it': 'Italiano', 'ru': 'Russo', 'zh': 'Chinês', 'cn': 'Chinês'
}

# Define a variável global que usaremos na Célula 6
nome_idioma_extenso = mapa_idiomas.get(lingua_input, lingua_input) # Assign here to ensure it's always defined

if lingua_input not in mapa_idiomas.keys():
  print(f'A língua colocada: "{lingua_input}" é invalida, rode a célula novamente.')
  # articles_list remains empty, preventing further steps with invalid language
else:
  # --- 3. Inicialização e Busca ---
  google_news = GNews(language=lingua_input, country=pais_input, max_results=5)
  # Garante que a chave definida na Célula 2 seja usada
  google_news.api_key = os.environ.get("GNEWS_API_KEY")

  print(f"\n🔍 Buscando as top 5 notícias em {nome_idioma_extenso} no país '{pais_input.upper()}'...")

  try:
      # Obtém as notícias
      articles_list = google_news.get_top_news()

      print(f"\n✅ Sucesso! Encontrados {len(articles_list)} artigos.")

      # Exibe o título do primeiro apenas para conferência
      if articles_list:
          print(f"Exemplo: {articles_list[0].get('title')}")
      else:
          print("⚠️ Nenhum artigo encontrado. Verifique se o código do país está correto ou sua cota de API.")

  except Exception as e:
      print(f"❌ Erro ao buscar notícias: {e}")
      print("Dica: Verifique se a API Key do GNews na Célula 2 está correta.")
      articles_list = [] # Ensure articles_list is empty on error

--- 🌍 Configuração da Busca de Notícias ---
Digite o código do país (ex: 'us', 'jp', 'br', 'fr') [padrão: us]: us
Digite o código do idioma (ex: 'en', 'ja', 'pt', 'fr') [padrão: en]: en

🔍 Buscando as top 5 notícias em Inglês no país 'US'...

✅ Sucesso! Encontrados 5 artigos.
Exemplo: Live Updates: Iran war escalates, energy prices spike after Israeli strike on South Pars gas field - CBS News


*Definição do LLM e Pydantic*

In [33]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List

# --- 1. Definição do LLM (Atualizado para Gemini 2.5) ---

# O modelo 1.5 foi descontinuado. Usamos agora o 'gemini-2.5-flash'.
# Este é o modelo padrão atual, rápido e dentro da cota gratuita.
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Verifica o nome do modelo
if hasattr(llm, 'model_name'):
    print(f"LLM inicializado: {llm.model_name}")
else:
    print(f"LLM inicializado: {llm.model}")


# --- 2. Definição do Schema de Saída (Pydantic) ---
class AnaliseArtigo(BaseModel):
    """Classe Pydantic para estruturar a análise do artigo."""
    titulo_traduzido: str = Field(description="Título do artigo, traduzido para o Português (Brasil).")
    resumo_traduzido: str = Field(description="Descrição/resumo do artigo, traduzido para o Português (Brasil).")
    sentimento: str = Field(description="O sentimento geral do artigo (Ex: 'Positivo', 'Negativo', 'Neutro').")
    entidades_principais: List[str] = Field(description="Uma lista de 3 a 5 entidades nomeadas (pessoas, organizações, locais) ou temas-chave mencionados.")
    idioma_original: str = Field(description="O idioma original do artigo (Ex: 'Japonês').")

print("\nSchema e Modelo (Gemini 2.5) definidos com sucesso.")

LLM inicializado: gemini-2.5-flash

Schema e Modelo (Gemini 2.5) definidos com sucesso.


*Criação do 'Mega-Chain'*

In [34]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# --- 1. Criação do Parser ---
# O parser usará nosso modelo Pydantic 'AnaliseArtigo' definido na Célula 4
parser = JsonOutputParser(pydantic_object=AnaliseArtigo)

# --- 2. Criação do Prompt Template ---
# Note que {idioma_origem} já é uma variável aqui, então aceita qualquer coisa!
template_string = """
Você é um analista de mídia internacional sênior, fluente em {idioma_origem} e Português (Brasil).
Sua tarefa é analisar o artigo de notícia a seguir e fornecer um relatório estruturado.

{instrucoes_de_formato}

---
Artigo de Notícia ({idioma_origem}):
Título: {titulo_original}
Descrição: {descricao_original}
---

Analise o artigo e forneça a saída no formato JSON solicitado.
"""

prompt = ChatPromptTemplate.from_template(
    template_string,
    partial_variables={"instrucoes_de_formato": parser.get_format_instructions()}
)

# --- 3. Criação da Chain (Pipeline) ---
# A chain conecta o Prompt -> LLM -> Parser
analysis_chain = prompt | llm | parser

print("✅ Pipeline (Chain) construído com sucesso!")
print("\n--- Exemplo de como o Prompt será enviado ao LLM (Prévia) ---")

# AJUSTE AQUI: Usamos o idioma escolhido na Célula 3 para o exemplo de visualização
exemplo_idioma = "Japonês" # Fallback
if 'nome_idioma_extenso' in globals():
    exemplo_idioma = nome_idioma_extenso

print(prompt.format(
    idioma_origem=exemplo_idioma,
    titulo_original="Exemplo de Título da Notícia",
    descricao_original="Exemplo do corpo da notícia..."
))

✅ Pipeline (Chain) construído com sucesso!

--- Exemplo de como o Prompt será enviado ao LLM (Prévia) ---
Human: 
Você é um analista de mídia internacional sênior, fluente em Inglês e Português (Brasil).
Sua tarefa é analisar o artigo de notícia a seguir e fornecer um relatório estruturado.

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": 

*Execução do Pipeline*

In [35]:
import time

# --- 1. Configuração do Contexto ---
# Tenta pegar o idioma definido na Célula 3.
# Se a célula 3 não foi rodada corretamente, usa um padrão de segurança.
try:
    idioma_contexto = nome_idioma_extenso
except NameError:
    idioma_contexto = "Inglês" # Fallback
    print("⚠️ Aviso: Variável de idioma não encontrada. Assumindo 'Inglês'.")

resultados_finais = []

print(f"🚀 Iniciando pipeline de análise para {len(articles_list)} artigos em '{idioma_contexto}'...")

# --- 2. Loop de Processamento ---
for i, article in enumerate(articles_list):
    print(f"\nProcessando Artigo {i+1}/{len(articles_list)}...")

    # Prepara o input para a chain com os dados da API GNews
    input_data = {
        "idioma_origem": idioma_contexto,
        "titulo_original": article.get('title', ''),
        "descricao_original": article.get('description', '')
    }

    try:
        # Invoca a chain (Prompt -> LLM -> Parser)
        resultado = analysis_chain.invoke(input_data)

        # Adiciona metadados extras (URL) ao resultado final
        resultado['url_original'] = article.get('url', '')

        # Salva na lista de resultados
        resultados_finais.append(resultado)
        print(f"✅ Artigo {i+1} analisado com sucesso.")

        # Pausa respeitosa para a API (evita erros de cota)
        time.sleep(1)

    except Exception as e:
        print(f"⚠️ Falha inicial no Artigo {i+1}: {e}")
        # Lógica de Retry (Tenta mais uma vez em caso de erro temporário)
        try:
            time.sleep(3)
            print("🔄 Tentando novamente...")
            resultado = analysis_chain.invoke(input_data)
            resultado['url_original'] = article.get('url', '')
            resultados_finais.append(resultado)
            print(f"✅ Artigo {i+1} analisado com sucesso na segunda tentativa.")
        except Exception as e_retry:
            print(f"❌ Falha definitiva no Artigo {i+1}: {e_retry}")

print(f"\n🏁 Análise concluída! {len(resultados_finais)} artigos processados com sucesso.")

🚀 Iniciando pipeline de análise para 5 artigos em 'Inglês'...

Processando Artigo 1/5...
✅ Artigo 1 analisado com sucesso.

Processando Artigo 2/5...
⚠️ Falha inicial no Artigo 2: Error calling model 'gemini-2.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key expired. Please renew the API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key expired. Please renew the API key.'}]}}
🔄 Tentando novamente...
❌ Falha definitiva no Artigo 2: Error calling model 'gemini-2.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key expired. Please renew the API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.

*Apresentação dos resultados*

In [36]:
from IPython.display import display, Markdown, HTML
from collections import Counter

if not resultados_finais:
    display(Markdown("### ⚠️ Nenhuma análise foi concluída com sucesso."))
    display(Markdown("Verifique se a Célula 6 rodou até o fim e gerou dados."))
else:
    # --- Cabeçalho do Relatório ---
    display(Markdown("# 🌎 Relatório de Mídia Internacional: Japão"))
    display(Markdown(f"Resumo das {len(resultados_finais)} principais notícias do dia."))

    # --- Preparação das Estatísticas ---
    # Agregando todos os sentimentos e entidades em listas únicas
    todos_sentimentos = [r['sentimento'] for r in resultados_finais]

    # Garante que entidades seja uma lista (segurança extra)
    todas_entidades = []
    for r in resultados_finais:
        ents = r.get('entidades_principais', [])
        if isinstance(ents, list):
            todas_entidades.extend(ents)
        elif isinstance(ents, str):
            todas_entidades.append(ents)

    # Contagem de sentimentos
    sentimento_geral = {
        "Positivo": todos_sentimentos.count("Positivo"),
        "Negativo": todos_sentimentos.count("Negativo"),
        "Neutro": todos_sentimentos.count("Neutro")
    }

    # Contagem de entidades (Top 5)
    entidades_comuns = Counter(todas_entidades).most_common(5)

    # --- Painel de Síntese (Dashboard) ---
    display(Markdown("---"))
    display(Markdown("### 📈 Síntese Geral"))

    # Exibe contagem de sentimentos
    display(Markdown(f"**Sentimento Geral:** 🟢 Positivo ({sentimento_geral['Positivo']}) | 🔴 Negativo ({sentimento_geral['Negativo']}) | ⚪ Neutro ({sentimento_geral['Neutro']})"))

    # Exibe top entidades
    if entidades_comuns:
        texto_entidades = ', '.join([f'{ent[0]} ({ent[1]}x)' for ent in entidades_comuns])
        display(Markdown(f"**Tópicos/Entidades mais Mencionados:** {texto_entidades}"))
    else:
        display(Markdown("**Tópicos/Entidades:** Não foi possível extrair padrões claros."))

    display(Markdown("---"))


    # --- Detalhe dos Artigos (Cards) ---
    display(Markdown("### 📰 Artigos Analisados"))

    for i, res in enumerate(resultados_finais):

        # Define a cor do sentimento para o visual
        s = res.get('sentimento', 'Neutro')
        if s == 'Positivo':
            cor_sentimento = 'green'
        elif s == 'Negativo':
            cor_sentimento = 'red'
        else:
            cor_sentimento = 'gray'

        # Garante que a lista de entidades vire uma string bonita
        ents_list = res.get('entidades_principais', [])
        if isinstance(ents_list, list):
            ents_str = ', '.join(ents_list)
        else:
            ents_str = str(ents_list)

        # Monta o HTML para exibição (Card estilo Dashboard)
        html_output = f"""
        <div style='border: 1px solid #e0e0e0; border-radius: 8px; padding: 20px; margin-bottom: 15px; background-color: #f9f9f9; font-family: sans-serif;'>

            <h3 style='margin-top:0; color: #333;'>{i+1}. {res.get('titulo_traduzido', 'Sem Título')}</h3>

            <div style='margin-bottom: 10px;'>
                <span style='background-color: {cor_sentimento}; color: white; padding: 4px 8px; border-radius: 4px; font-size: 12px; font-weight: bold;'>
                    {s.upper()}
                </span>
                <span style='color: #666; font-size: 12px; margin-left: 10px;'>
                    Idioma Original: {res.get('idioma_original', 'Desconhecido')}
                </span>
            </div>

            <p style='color: #444; line-height: 1.5;'><strong>Resumo:</strong> {res.get('resumo_traduzido', 'Sem resumo.')}</p>

            <p style='font-size: 13px; color: #555;'>
                <strong>🔍 Entidades/Temas:</strong> <code>{ents_str}</code>
            </p>

            <div style='margin-top: 15px;'>
                <a href='{res.get('url_original', '#')}' target='_blank' style='text-decoration: none; color: #0066cc; font-weight: bold;'>
                    🔗 Ler artigo original &rarr;
                </a>
            </div>

        </div>
        """
        display(HTML(html_output))

# 🌎 Relatório de Mídia Internacional: Japão

Resumo das 4 principais notícias do dia.

---

### 📈 Síntese Geral

**Sentimento Geral:** 🟢 Positivo (0) | 🔴 Negativo (3) | ⚪ Neutro (1)

**Tópicos/Entidades mais Mencionados:** Irã (2x), Israel (1x), Campo de Gás South Pars (1x), Preços de Energia (1x), Conflito no Oriente Médio (1x)

---

### 📰 Artigos Analisados